# Formula E — video verifier check (video-direct)

Validates the shipping approach: on a telemetry stop, point Gemini straight at each mosaic in the bucket and pass `videoMetadata` start/end offsets so it decodes only that window — **no download, no frame extraction, no warm-up.** It sweeps all six camera groups and judges the track state (persistent **blockage** vs **cleared**).

**Run-all** sets up the Gemini client and checks the three known cases — expect **blocked / cleared / blocked** for 693 / 1373 / 1780. Needs this project's ADC with read on `MOSAICS_BASE`.

## 1 · Config

In [ ]:
PROJECT_ID = ""     # e.g. "qwiklabs-gcp-04-..."; blank = notebook default project
# Where the six group mosaics live (each is <group_id>.mp4). Use your project bucket
# (same-region reads are free) rather than the shared class bucket.
MOSAICS_BASE = "gs://class-demo/formula-e/r10/mosaics"
#   ... or:  f"gs://{PROJECT_ID}-fe-mosaics/mosaics"
GEM_MODEL = "gemini-3.5-flash"
print("mosaics base:", MOSAICS_BASE)

## 2 · Gemini client (Vertex + ADC)

In [ ]:
import json, re, time
try:
    from google import genai
    from google.genai import types
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'google-genai'], check=True)
    from google import genai
    from google.genai import types
import google.auth

_proj = PROJECT_ID or google.auth.default()[1]
GEM = genai.Client(vertexai=True, project=_proj, location='global')
print('Gemini ready via Vertex project', _proj, 'model', GEM_MODEL)

def _parse(text):
    s = (text or '').strip()
    a, b = s.find('{'), s.rfind('}')
    if a != -1 and b > a:
        try: return json.loads(s[a:b+1])
        except Exception: pass
    return {}

## 3 · Camera groups
The six 2x2 mosaics, each four track-consecutive cameras (TL, TR, BL, BR).

In [ ]:
GROUPS = {
    'grp_01_cam01_cam02_cam03_cam04': ['Cam01','Cam02','Cam03','Cam04'],
    'grp_02_cam05_cam06_cam07_cam08': ['Cam05','Cam06','Cam07','Cam08'],
    'grp_03_cam09_cam10_cam11_cam12': ['Cam09','Cam10','Cam11','Cam12'],
    'grp_04_cam13_cam14_cam15_cam16': ['Cam13','Cam14','Cam15','Cam16'],
    'grp_05_cam17_cam18_cam19_cam20': ['Cam17','Cam18','Cam19','Cam20'],
    'grp_06_cam21_cam22_cam23_cam24': ['Cam21','Cam22','Cam23','Cam24'],
}
print(len(GROUPS), 'groups')

## 4 · The verifier — hand Gemini a gs:// slice
For each group: a `gs://` video Part with start/end offsets (the window around the stop) + a grounded persistence prompt. Judges **track state**, not car identity; sweeps all groups and takes the strongest blockage.

In [ ]:
def verify_video_direct(t, lead=10, tail=50):
    """Persistence sweep over gs:// video slices (no download/extraction)."""
    start, end = max(0, t - lead), t + tail
    print(f"\n### @ t={t}s  (gs:// slice {start}s-{end}s, all {len(GROUPS)} groups)")
    verdicts = []
    for gid, cams in GROUPS.items():
        uri = f"{MOSAICS_BASE}/{gid}.mp4"
        prompt = (
            'You are a race-control video verifier deciding whether a SAFETY CAR is warranted.\n'
            f'Telemetry flagged a car possibly stopped near here around race time ~{t}s.\n'
            f'This is a ~{end-start}s CCTV clip - a 2x2 mosaic of four cameras: '
            f'TL={cams[0]}, TR={cams[1]}, BL={cams[2]}, BR={cams[3]} - covering that moment.\n'
            'Watch the clip and judge the state by the END:\n'
            '- A car STILL stopped/stranded on or beside the racing line at the end (persistent '
            'obstruction, maybe marshals/recovery): blockage=true, cleared=false.\n'
            '- A car appeared but DROVE AWAY / was recovered / line clear by the end: '
            'blockage=false, cleared=true.\n'
            '- No stopped car at any point: blockage=false, cleared=false.\n'
            'Do NOT identify the car number; judge only the track state. Note if other cars are moving.\n'
            'JSON: {"blockage": bool, "cleared": bool, "panel": "TL|TR|BL|BR|none", "feed_live": bool, "what_you_see": str, "confidence": number}')
        try:
            vpart = types.Part(
                file_data=types.FileData(file_uri=uri, mime_type='video/mp4'),
                video_metadata=types.VideoMetadata(start_offset=f'{start}s', end_offset=f'{end}s'))
            resp = GEM.models.generate_content(
                model=GEM_MODEL,
                contents=[types.Content(role='user', parts=[vpart, types.Part(text=prompt)])],
                config=types.GenerateContentConfig(temperature=0.2, response_mime_type='application/json'))
            d = _parse(resp.text)
        except Exception as e:
            print(f"  {gid:34} ERROR: {e}"); continue
        state = 'BLOCKED' if d.get('blockage') else ('cleared' if d.get('cleared') else '-')
        print(f"  {gid:34} {state:8} panel={str(d.get('panel','?')):5} conf={d.get('confidence','?')}")
        if d.get('blockage') or d.get('cleared'):
            print(f"        -> {str(d.get('what_you_see',''))[:150]}")
        verdicts.append((gid, d))
    blocked = [g for g, d in verdicts if d.get('blockage')]
    print('  VERDICT: ' + ('BLOCKAGE in ' + ', '.join(blocked) if blocked else 'no blockage (cleared)'))
    return verdicts

## 5 · Validate — three known cases
Expect **blocked / cleared / blocked**. (Ground truth: #7 Günther and #23 Fenestraz retired; Evans #9 had a big decel at 1373s but finished 6th.)

In [ ]:
_ = verify_video_direct(693)     # Gunther @T-? -> BLOCKED
_ = verify_video_direct(1373)    # Evans          -> CLEARED
_ = verify_video_direct(1780)    # Mortara/#48    -> BLOCKED

## 6 · Timing
Real per-verify latency (six groups, one stop) — no download/extract, so this is what the live agent pays.

In [ ]:
t0 = time.time(); _ = verify_video_direct(693)
print(f"\nelapsed: {time.time()-t0:.1f}s for 6 groups")